# WeatherGPT — NLU Training (Intent + Slot Extraction)
**SIH26068 · ML/NLP layer — replaces keyword-matched/canned responses with real language understanding**

## What this notebook builds and why
Your original mock chat logic matched a handful of hardcoded keywords ("patna", "cyclone",
"rain"). That only ever answers questions it was explicitly written for. This notebook trains
a real **joint intent classification + slot-filling model** — the same architecture pattern
used by production voice assistants (Alexa/Google Assistant NLU) — so the bot understands
*any phrasing* of a weather question, not a fixed list.

**Important scoping note, stated plainly:** this makes WeatherGPT robust to arbitrary
phrasing of *weather-related* questions ("will it pour in Chennai this evening?", "should I
carry an umbrella tomorrow?", "kal Patna mein baarish hogi kya?" all resolve to the same
intent+slots). It does **not** make WeatherGPT a general-purpose chatbot that answers
anything under the sun — an `out_of_scope` intent is trained specifically to detect
non-weather questions and decline them gracefully rather than hallucinate an answer. That's
a deliberate, honest design choice, not a limitation to hide: a disaster-alert tool that
confidently answers trivia questions with made-up facts is worse than one that says
"I can only help with weather" for those.

## Pipeline
1. Generate a large, template-based synthetic training set (since no ready-made labeled
   "Indian weather query" dataset exists publicly) covering many intents, slot types,
   phrasings, and a starter set of Hindi examples.
2. Fine-tune a joint intent+slot BERT model (single backbone, two output heads) —
   more efficient than two separate models and the standard approach for this task.
3. Evaluate: intent accuracy, per-class slot F1.
4. Export the model + a `parse()` inference function used by `serve_nlu.py`
   (in the accompanying files) to actually answer chat queries with real weather data,
   not templates.


## 1 — Setup

In [ ]:
!pip -q install transformers datasets seqeval accelerate --upgrade

import torch, random, json, re
import numpy as np
from collections import Counter
from transformers import AutoTokenizer, AutoModel
from seqeval.metrics import classification_report as seq_classification_report, f1_score as seq_f1_score
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

# Multilingual so the same model handles English + a starter set of Hindi phrasings —
# swap for 'ai4bharat/indic-bert' if you want stronger coverage across more Indian
# languages later (larger vocab, trained specifically on Indic text).
MODEL_NAME = "distilbert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


## 2 — Define the label schema

**Intents** cover every question type WeatherGPT should route to a real answer, plus
conversational and out-of-scope catch-alls.

**Slots** (BIO-tagged) extract the pieces of information needed to actually fetch the right
data: `LOC` (a place), `DATETIME` (when), `ATTR` (which weather attribute — rain, wind,
temperature, etc).

In [ ]:
INTENTS = [
    "current_weather", "forecast", "rain_probability", "temperature_query",
    "wind_query", "humidity_query", "alert_check", "cyclone_status", "flood_risk",
    "clothing_advice", "travel_advice", "general_climate",
    "greeting", "thanks", "goodbye", "out_of_scope",
]
INTENT2ID = {intent: i for i, intent in enumerate(INTENTS)}
ID2INTENT = {i: intent for intent, i in INTENT2ID.items()}

SLOT_TYPES = ["LOC", "DATETIME", "ATTR"]
SLOT_LABELS = ["O"] + [f"{prefix}-{t}" for t in SLOT_TYPES for prefix in ("B", "I")]
SLOT2ID = {label: i for i, label in enumerate(SLOT_LABELS)}
ID2SLOT = {i: label for label, i in SLOT2ID.items()}

print(f"{len(INTENTS)} intents, {len(SLOT_LABELS)} slot labels: {SLOT_LABELS}")


## 3 — Generate the synthetic training set

Real production NLU systems are bootstrapped this way too (template generation, then
refined with real logged queries once the product is live) — there's no shame in synthetic
data for a first version; the shame would be shipping a bot that can only answer 15
hardcoded questions. Members 3/4: once this app collects real user queries, mixing those
into this generator's output (as new templates or as additional labeled examples) is the
single highest-value next step.

In [ ]:
LOCATIONS = [
    "Delhi", "Mumbai", "Patna", "Chennai", "Kolkata", "Bengaluru", "Hyderabad", "Jaipur",
    "Lucknow", "Ahmedabad", "Pune", "Surat", "Kanpur", "Nagpur", "Indore", "Bhopal",
    "Visakhapatnam", "Coimbatore", "Kochi", "Guwahati", "Shimla", "Srinagar", "Leh", "Goa",
    "coastal Odisha", "Bhubaneswar", "North India", "South India", "the Western Ghats",
    "Ladakh", "Darjeeling", "Varanasi", "Amritsar", "Chandigarh", "Ranchi", "Raipur",
    "Dehradun", "Thiruvananthapuram", "Mysuru", "Nashik",
]

DATETIMES = [
    "today", "tomorrow", "tonight", "this evening", "this weekend", "next Monday",
    "in 3 days", "this week", "next week", "on Friday", "this afternoon", "right now",
    "tomorrow morning", "over the next few days",
]

ATTRS = {
    "rain": ["rain", "rainfall", "showers", "downpour"],
    "temperature": ["temperature", "heat", "how hot", "how cold"],
    "wind": ["wind", "wind speed", "gusts"],
    "humidity": ["humidity"],
    "snow": ["snow", "snowfall"],
    "storm": ["storm", "thunderstorm"],
}

# Each template's {loc}/{date}/{attr} placeholders get BIO-tagged automatically based on
# where they land in the generated sentence — see the tagging function below.
TEMPLATES = {
    "current_weather": [
        "what's the weather like in {loc} right now",
        "how's the weather in {loc}",
        "tell me the current weather for {loc}",
        "is it hot in {loc} today",
    ],
    "forecast": [
        "what will the weather be like in {loc} {date}",
        "give me the forecast for {loc} {date}",
        "what's {loc} looking like {date}",
        "forecast for {loc} {date} please",
    ],
    "rain_probability": [
        "will it rain in {loc} {date}",
        "is it going to rain in {loc} {date}",
        "chances of {attr} in {loc} {date}",
        "should I expect {attr} in {loc} {date}",
        "kya {loc} mein {date} baarish hogi",
    ],
    "temperature_query": [
        "what's the temperature in {loc} {date}",
        "how hot will it be in {loc} {date}",
        "{loc} ka tapmaan kya hai {date}",
    ],
    "wind_query": [
        "how windy is it in {loc} {date}",
        "what's the wind speed in {loc} {date}",
    ],
    "humidity_query": [
        "what's the humidity in {loc} {date}",
        "is it humid in {loc} {date}",
    ],
    "alert_check": [
        "is there any weather alert for {loc}",
        "any warnings for {loc} {date}",
        "are there severe weather alerts in {loc}",
        "{loc} mein koi chetavani hai kya",
    ],
    "cyclone_status": [
        "is there a cyclone warning for {loc}",
        "any cyclone alerts near {loc}",
        "is a cyclone approaching {loc}",
    ],
    "flood_risk": [
        "is there a flood risk in {loc} {date}",
        "any flood warnings for {loc}",
    ],
    "clothing_advice": [
        "what should I wear in {loc} {date}",
        "do I need a jacket in {loc} {date}",
        "should I carry an umbrella in {loc} {date}",
    ],
    "travel_advice": [
        "is it safe to travel to {loc} {date}",
        "should I travel to {loc} {date} given the weather",
    ],
    "general_climate": [
        "what's the climate like in {loc}",
        "how's the weather usually in {loc}",
    ],
    "greeting": ["hi", "hello", "hey there", "namaste", "good morning"],
    "thanks": ["thanks", "thank you", "thanks a lot", "shukriya"],
    "goodbye": ["bye", "goodbye", "see you", "alvida"],
    "out_of_scope": [
        "what's the capital of France", "tell me a joke", "who is the prime minister",
        "what's 2 plus 2", "recommend me a movie", "how do I cook rice",
        "what's the stock price of Tesla", "translate hello to French",
        "who won the cricket match yesterday", "what time is it in New York",
    ],
}


def tag_template(template, loc=None, date=None, attr=None):
    """Fills a template and produces (tokens, bio_tags) with correct BIO alignment
    for each filled slot span."""
    text = template
    fills = {"{loc}": (loc, "LOC"), "{date}": (date, "DATETIME"), "{attr}": (attr, "ATTR")}

    # Build token list + tags by walking the template piece by piece.
    tokens, tags = [], []
    pattern = re.compile(r"(\{loc\}|\{date\}|\{attr\})")
    parts = pattern.split(template)
    for part in parts:
        if part in fills:
            value, slot_type = fills[part]
            if value is None:
                continue
            words = value.split()
            for i, w in enumerate(words):
                tokens.append(w)
                tags.append(f"{'B' if i == 0 else 'I'}-{slot_type}")
        elif part.strip():
            for w in part.strip().split():
                tokens.append(w)
                tags.append("O")
    return tokens, tags


def generate_examples(n_per_intent=250):
    examples = []
    for intent, templates in TEMPLATES.items():
        for _ in range(n_per_intent):
            template = random.choice(templates)
            needs_loc = "{loc}" in template
            needs_date = "{date}" in template
            needs_attr = "{attr}" in template

            loc = random.choice(LOCATIONS) if needs_loc else None
            date = random.choice(DATETIMES) if needs_date else None
            attr_key = random.choice(list(ATTRS.keys())) if needs_attr else None
            attr = random.choice(ATTRS[attr_key]) if attr_key else None

            tokens, tags = tag_template(template, loc, date, attr)
            if not tokens:
                continue
            examples.append({"tokens": tokens, "tags": tags, "intent": intent})
    random.shuffle(examples)
    return examples


all_examples = generate_examples(n_per_intent=220)
print(f"Generated {len(all_examples)} examples")
print("Intent distribution:", Counter(e['intent'] for e in all_examples))
print("\nSample:")
for ex in all_examples[:3]:
    print(ex)


In [ ]:
# Train/val/test split, stratified by intent so rare-ish intents (out_of_scope etc.)
# still show up in every split.
from sklearn.model_selection import train_test_split

intents_list = [e['intent'] for e in all_examples]
train_examples, temp_examples = train_test_split(
    all_examples, test_size=0.3, stratify=intents_list, random_state=SEED
)
temp_intents = [e['intent'] for e in temp_examples]
val_examples, test_examples = train_test_split(
    temp_examples, test_size=0.5, stratify=temp_intents, random_state=SEED
)
print(f"Train: {len(train_examples)}  Val: {len(val_examples)}  Test: {len(test_examples)}")


## 4 — Tokenize and align slot labels to subword tokens

In [ ]:
MAX_LEN = 32

def encode_examples(examples):
    all_input_ids, all_attn_masks, all_slot_labels, all_intent_labels = [], [], [], []

    for ex in examples:
        encoding = tokenizer(
            ex['tokens'], is_split_into_words=True, truncation=True,
            max_length=MAX_LEN, padding='max_length', return_tensors=None,
        )
        word_ids = encoding.word_ids()
        label_ids = []
        prev_word_id = None
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)  # special tokens ([CLS]/[SEP]/padding) — ignored in loss
            elif word_id != prev_word_id:
                label_ids.append(SLOT2ID[ex['tags'][word_id]])
            else:
                # subword continuation of a word already labeled — also ignored,
                # standard practice for token classification with subword tokenizers
                label_ids.append(-100)
            prev_word_id = word_id

        all_input_ids.append(encoding['input_ids'])
        all_attn_masks.append(encoding['attention_mask'])
        all_slot_labels.append(label_ids)
        all_intent_labels.append(INTENT2ID[ex['intent']])

    return {
        'input_ids': torch.tensor(all_input_ids),
        'attention_mask': torch.tensor(all_attn_masks),
        'slot_labels': torch.tensor(all_slot_labels),
        'intent_labels': torch.tensor(all_intent_labels),
    }

train_enc = encode_examples(train_examples)
val_enc = encode_examples(val_examples)
test_enc = encode_examples(test_examples)
print("Encoded shapes:", train_enc['input_ids'].shape)


## 5 — Model: joint intent + slot BERT

One shared transformer backbone, two heads:
- **Intent head**: reads the `[CLS]` token's pooled representation → classifies the
  whole utterance into one of the 16 intents.
- **Slot head**: reads every token's hidden state → classifies each token into a BIO slot
  label.

Training both jointly (shared backbone, summed loss) is more efficient than two separate
models and lets the two tasks share useful representations — the standard architecture for
this kind of task-oriented NLU (the pattern behind most commercial voice assistants).

In [ ]:
class JointIntentSlotModel(torch.nn.Module):
    def __init__(self, model_name, num_intents, num_slots, dropout=0.1):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size
        self.dropout = torch.nn.Dropout(dropout)
        self.intent_classifier = torch.nn.Linear(hidden_size, num_intents)
        self.slot_classifier = torch.nn.Linear(hidden_size, num_slots)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state       # (batch, seq_len, hidden) — for slots
        pooled_output = sequence_output[:, 0, :]           # [CLS] token — for intent

        intent_logits = self.intent_classifier(self.dropout(pooled_output))
        slot_logits = self.slot_classifier(self.dropout(sequence_output))
        return intent_logits, slot_logits


model = JointIntentSlotModel(MODEL_NAME, len(INTENTS), len(SLOT_LABELS)).to(device)
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")


## 6 — Training loop

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

def make_loader(enc, batch_size, shuffle):
    ds = TensorDataset(enc['input_ids'], enc['attention_mask'], enc['slot_labels'], enc['intent_labels'])
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

BATCH_SIZE = 32
train_loader = make_loader(train_enc, BATCH_SIZE, shuffle=True)
val_loader = make_loader(val_enc, BATCH_SIZE, shuffle=False)
test_loader = make_loader(test_enc, BATCH_SIZE, shuffle=False)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)
intent_loss_fn = torch.nn.CrossEntropyLoss()
slot_loss_fn = torch.nn.CrossEntropyLoss(ignore_index=-100)

EPOCHS = 8
SLOT_LOSS_WEIGHT = 1.0  # tune if slot F1 lags noticeably behind intent accuracy


def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, total_intent_correct, total_examples = 0.0, 0, 0

    with torch.set_grad_enabled(train):
        for input_ids, attn_mask, slot_labels, intent_labels in loader:
            input_ids, attn_mask = input_ids.to(device), attn_mask.to(device)
            slot_labels, intent_labels = slot_labels.to(device), intent_labels.to(device)

            intent_logits, slot_logits = model(input_ids, attn_mask)

            intent_loss = intent_loss_fn(intent_logits, intent_labels)
            slot_loss = slot_loss_fn(slot_logits.view(-1, len(SLOT_LABELS)), slot_labels.view(-1))
            loss = intent_loss + SLOT_LOSS_WEIGHT * slot_loss

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * input_ids.size(0)
            total_intent_correct += (intent_logits.argmax(-1) == intent_labels).sum().item()
            total_examples += input_ids.size(0)

    return total_loss / total_examples, total_intent_correct / total_examples


history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0

for epoch in range(EPOCHS):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)

    history['train_loss'].append(train_loss); history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc); history['val_acc'].append(val_acc)

    print(f"Epoch {epoch+1}/{EPOCHS} | train_loss={train_loss:.4f} train_acc={train_acc:.3f} | "
          f"val_loss={val_loss:.4f} val_acc={val_acc:.3f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_nlu_model.pt')

print(f"\nBest validation intent accuracy: {best_val_acc:.3f}")


In [ ]:
plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='train'); plt.plot(history['val_loss'], label='val')
plt.title('Loss'); plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='train'); plt.plot(history['val_acc'], label='val')
plt.title('Intent accuracy'); plt.legend()
plt.tight_layout(); plt.show()


## 7 — Evaluate on the held-out test set

In [ ]:
model.load_state_dict(torch.load('best_nlu_model.pt'))
model.eval()

all_intent_true, all_intent_pred = [], []
all_slot_true, all_slot_pred = [], []

with torch.no_grad():
    for input_ids, attn_mask, slot_labels, intent_labels in test_loader:
        input_ids, attn_mask = input_ids.to(device), attn_mask.to(device)
        intent_logits, slot_logits = model(input_ids, attn_mask)

        all_intent_true.extend(intent_labels.numpy())
        all_intent_pred.extend(intent_logits.argmax(-1).cpu().numpy())

        slot_preds = slot_logits.argmax(-1).cpu().numpy()
        for i in range(slot_labels.size(0)):
            true_seq, pred_seq = [], []
            for j in range(slot_labels.size(1)):
                if slot_labels[i, j].item() != -100:
                    true_seq.append(ID2SLOT[slot_labels[i, j].item()])
                    pred_seq.append(ID2SLOT[slot_preds[i, j]])
            all_slot_true.append(true_seq)
            all_slot_pred.append(pred_seq)

print("=== Intent classification report ===")
print(classification_report(all_intent_true, all_intent_pred, target_names=INTENTS, digits=3))

print("\n=== Slot filling report ===")
print(seq_classification_report(all_slot_true, all_slot_pred, digits=3))
print(f"\nOverall slot F1: {seq_f1_score(all_slot_true, all_slot_pred):.3f}")


In [ ]:
cm = confusion_matrix(all_intent_true, all_intent_pred)
plt.figure(figsize=(11, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=INTENTS, yticklabels=INTENTS)
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Intent confusion matrix')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()


## 8 — Inference function (this is what `serve_nlu.py` calls)

In [ ]:
def parse(text: str):
    """Parses a raw user utterance into intent + extracted slots.
    This is the function that replaces keyword-matching entirely — any phrasing
    of a supported question type routes correctly, not just the exact strings
    it was tested on above."""
    words = text.strip().split()
    encoding = tokenizer(
        words, is_split_into_words=True, truncation=True,
        max_length=MAX_LEN, padding='max_length', return_tensors='pt',
    )
    word_ids = encoding.word_ids()

    model.eval()
    with torch.no_grad():
        intent_logits, slot_logits = model(
            encoding['input_ids'].to(device), encoding['attention_mask'].to(device)
        )

    intent = ID2INTENT[intent_logits.argmax(-1).item()]
    intent_confidence = torch.softmax(intent_logits, -1).max().item()

    slot_preds = slot_logits.argmax(-1)[0].cpu().numpy()
    slots = {}
    current_type, current_words = None, []

    def flush():
        if current_type and current_words:
            slots.setdefault(current_type, []).append(" ".join(current_words))

    prev_word_id = None
    for i, word_id in enumerate(word_ids):
        if word_id is None or word_id == prev_word_id:
            continue
        label = ID2SLOT[slot_preds[i]]
        if label.startswith("B-"):
            flush()
            current_type = label[2:]
            current_words = [words[word_id]]
        elif label.startswith("I-") and current_type == label[2:]:
            current_words.append(words[word_id])
        else:
            flush()
            current_type, current_words = None, []
        prev_word_id = word_id
    flush()

    return {
        "intent": intent,
        "confidence": round(intent_confidence, 3),
        "location": slots.get("LOC", [None])[0],
        "datetime": slots.get("DATETIME", [None])[0],
        "attribute": slots.get("ATTR", [None])[0],
    }


# Quick sanity check with phrasings NOT in the exact template list — this is the
# real test of "does it generalize", not just "does it memorize the training set".
for q in [
    "hey will it be raining in chennai this weekend?",
    "do i need an umbrella tomorrow in mumbai",
    "any cyclone warnings near bhubaneswar right now",
    "what's the capital of france",
    "thanks a bunch!",
]:
    print(q, "->", parse(q))


## 9 — Export

Saves the model weights, tokenizer, and label mappings needed by `serve_nlu.py`.
Note: this is a transformer model (~135M parameters for DistilBERT) — realistically this
should be **backend-served**, not bundled on-device. Converting it to something mobile-sized
(ONNX Runtime Mobile + quantization, or distilling into a much smaller model) is a valid
future optimization, but out of scope for a 13-day hackathon timeline.

In [ ]:
import os

EXPORT_DIR = 'weathergpt_nlu_export'
os.makedirs(EXPORT_DIR, exist_ok=True)

torch.save(model.state_dict(), os.path.join(EXPORT_DIR, 'model.pt'))
tokenizer.save_pretrained(EXPORT_DIR)

with open(os.path.join(EXPORT_DIR, 'label_maps.json'), 'w') as f:
    json.dump({
        'intents': INTENTS,
        'slot_labels': SLOT_LABELS,
        'model_name': MODEL_NAME,
    }, f, indent=2)

import shutil as _shutil
_shutil.make_archive('weathergpt_nlu_export', 'zip', EXPORT_DIR)

from google.colab import files
files.download('weathergpt_nlu_export.zip')


## 10 — What's next for this model
- **Collect real logged queries** once the app is live (even from your own team testing
  the demo) and mix them into `TEMPLATES`/as new labeled examples — synthetic data gets
  you a strong first version, real usage data is what actually improves it further.
- **Expand Hindi (and other Indic language) coverage** — right now Hindi is a small
  starter set piggybacked onto a few templates. A genuinely multilingual product needs
  proportionally more native-language training examples, not just a token gesture.
- **Add more out-of-scope examples** as you discover questions people actually try that
  aren't weather-related — the more varied that class is, the more reliably the model
  declines gracefully instead of guessing.
- Watch the confusion matrix for intents that are semantically close (e.g.
  `rain_probability` vs `forecast` vs `travel_advice` — a "should I travel" question about
  rain could reasonably be tagged as any of the three). If the model is confusing intents
  a human would also find ambiguous, that's the templates' fault, not the model's — refine
  the label boundaries rather than just adding more data.
